# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [2]:
import pandas as pd

# Load the three datasets
data_1 = pd.read_csv(
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
)

data_2 = pd.read_csv(
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
)

data_3 = pd.read_csv(
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"
)

# Inspect dimensions and column names
for number, dataset in enumerate([data_1, data_2, data_3], start=1):
    print(f"Dataset {number} shape: {dataset.shape}")
    print(f"Dataset {number} columns: {dataset.columns.tolist()}\n")

Dataset 1 shape: (4008, 11)
Dataset 1 columns: ['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Vehicle Class', 'Total Claim Amount']

Dataset 2 shape: (996, 11)
Dataset 2 columns: ['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Total Claim Amount', 'Policy Type', 'Vehicle Class']

Dataset 3 shape: (7070, 11)
Dataset 3 columns: ['Customer', 'State', 'Customer Lifetime Value', 'Education', 'Gender', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Total Claim Amount', 'Vehicle Class']



In [3]:
# Standardize column names in the first two datasets
column_mapping = {
    "ST": "State",
    "GENDER": "Gender"
}

data_1 = data_1.rename(columns=column_mapping)
data_2 = data_2.rename(columns=column_mapping)

# Combine the three datasets vertically and reset the index
combined_data = pd.concat(
    [data_1, data_2, data_3],
    ignore_index=True
)

# Check the combined dataset
print(f"Combined dataset shape: {combined_data.shape}")
display(combined_data.head())

# Inspect data types and non-null counts before cleaning
combined_data.info()

Combined dataset shape: (12074, 11)


,Customer,State,Gender,Education,Customer Lifetime Value,Income,Monthly Premium Auto,Number of Open Complaints,Policy Type,Vehicle Class,Total Claim Amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


<class 'pandas.DataFrame'>
RangeIndex: 12074 entries, 0 to 12073
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Customer                   9137 non-null   str    
 1   State                      9137 non-null   str    
 2   Gender                     9015 non-null   str    
 3   Education                  9137 non-null   str    
 4   Customer Lifetime Value    9130 non-null   object 
 5   Income                     9137 non-null   float64
 6   Monthly Premium Auto       9137 non-null   float64
 7   Number of Open Complaints  9137 non-null   object 
 8   Policy Type                9137 non-null   str    
 9   Vehicle Class              9137 non-null   str    
 10  Total Claim Amount         9137 non-null   float64
dtypes: float64(3), object(2), str(6)
memory usage: 1.0+ MB


In [4]:
# Remove rows where every column is missing
clean_data = combined_data.dropna(how="all").copy()

# Reset the index after removing empty rows
clean_data = clean_data.reset_index(drop=True)

# Report the number of removed rows
removed_rows = len(combined_data) - len(clean_data)

print(f"Completely empty rows removed: {removed_rows}")
print(f"Dataset shape after removing empty rows: {clean_data.shape}")

# Check remaining missing values
print("\nRemaining missing values per column:")
print(clean_data.isna().sum())

# Count fully duplicated rows without removing them yet
print(f"\nFully duplicated rows: {clean_data.duplicated().sum()}")

Completely empty rows removed: 2937
Dataset shape after removing empty rows: (9137, 11)

Remaining missing values per column:
Customer                       0
State                          0
Gender                       122
Education                      0
Customer Lifetime Value        7
Income                         0
Monthly Premium Auto           0
Number of Open Complaints      0
Policy Type                    0
Vehicle Class                  0
Total Claim Amount             0
dtype: int64

Fully duplicated rows: 3


In [6]:
# Remove fully duplicated rows
clean_data = clean_data.drop_duplicates().reset_index(drop=True)

print(f"Dataset shape after removing duplicates: {clean_data.shape}")

# Inspect categorical labels
categorical_columns = [
    "State", "Gender", "Education", "Policy Type", "Vehicle Class"
]

for column in categorical_columns:
    print(f"\n{column}:")
    print(clean_data[column].dropna().unique())

# Compare numeric fields stored in different formats across sources
columns_to_check = [
    "Customer Lifetime Value",
    "Number of Open Complaints"
]

for number, dataset in enumerate([data_1, data_2, data_3], start=1):
    print(f"\nDataset {number} examples:")
    for column in columns_to_check:
        examples = dataset[column].dropna().head(5).tolist()
        print(f"{column}: {examples}")

Dataset shape after removing duplicates: (9134, 11)

State:
<StringArray>
['Washington', 'Arizona', 'Nevada', 'California', 'Oregon', 'Cali', 'AZ',
 'WA']
Length: 8, dtype: str

Gender:
<StringArray>
['F', 'M', 'Femal', 'Male', 'female']
Length: 5, dtype: str

Education:
<StringArray>
['Master', 'Bachelor', 'High School or Below', 'College', 'Bachelors',
 'Doctor']
Length: 6, dtype: str

Policy Type:
<StringArray>
['Personal Auto', 'Corporate Auto', 'Special Auto']
Length: 3, dtype: str

Vehicle Class:
<StringArray>
['Four-Door Car',  'Two-Door Car',           'SUV',    'Luxury SUV',
    'Sports Car',    'Luxury Car']
Length: 6, dtype: str

Dataset 1 examples:
Customer Lifetime Value: ['697953.59%', '1288743.17%', '764586.18%', '536307.65%', '825629.78%']
Number of Open Complaints: ['1/0/00', '1/0/00', '1/0/00', '1/0/00', '1/0/00']

Dataset 2 examples:
Customer Lifetime Value: ['323912.47%', '462680.11%', '899704.02%', '2580706.30%', '380812.21%']
Number of Open Complaints: ['1/0/00', 

In [7]:
# Standardize state names
clean_data["State"] = clean_data["State"].str.strip().replace({
    "Cali": "California",
    "AZ": "Arizona",
    "WA": "Washington"
})

# Standardize gender labels while preserving missing values
clean_data["Gender"] = clean_data["Gender"].str.strip().replace({
    "F": "Female",
    "Femal": "Female",
    "female": "Female",
    "M": "Male"
})

# Standardize education labels
clean_data["Education"] = clean_data["Education"].str.strip().replace({
    "Bachelors": "Bachelor"
})

# Verify the standardized categories
for column in ["State", "Gender", "Education"]:
    print(f"{column}: {clean_data[column].dropna().unique().tolist()}")

State: ['Washington', 'Arizona', 'Nevada', 'California', 'Oregon']
Gender: ['Female', 'Male']
Education: ['Master', 'Bachelor', 'High School or Below', 'College', 'Doctor']


In [8]:
# Inspect formats in the first two source datasets
for number, dataset in enumerate([data_1, data_2], start=1):
    print(f"Dataset {number}:")
    print("CLV:", dataset["Customer Lifetime Value"].dropna().head(3).tolist())
    print(
        "Complaints:",
        dataset["Number of Open Complaints"].dropna().unique().tolist()
    )

Dataset 1:
CLV: ['697953.59%', '1288743.17%', '764586.18%']
Complaints: ['1/0/00', '1/2/00', '1/1/00', '1/3/00', '1/5/00', '1/4/00']
Dataset 2:
CLV: ['323912.47%', '462680.11%', '899704.02%']
Complaints: ['1/0/00', '1/1/00', '1/3/00', '1/5/00', '1/2/00', '1/4/00']


In [9]:
# Identify CLV values stored with a percentage symbol
clv_text = clean_data["Customer Lifetime Value"].astype("string").str.strip()
percentage_mask = clv_text.str.endswith("%", na=False)

# Convert to numeric and rescale only percentage-formatted values
clv_numeric = pd.to_numeric(
    clv_text.str.replace("%", "", regex=False),
    errors="raise"
)

clean_data["Customer Lifetime Value"] = clv_numeric.where(
    ~percentage_mask,
    clv_numeric / 100
).astype("float64")

# Extract complaint counts from the encoded text format
complaints_text = (
    clean_data["Number of Open Complaints"]
    .astype("string")
    .str.strip()
    .str.replace(r"^1/(\d+)/00$", r"\1", regex=True)
)

clean_data["Number of Open Complaints"] = pd.to_numeric(
    complaints_text,
    errors="raise"
).astype("Int64")

# Verify the converted columns
converted_columns = [
    "Customer Lifetime Value",
    "Number of Open Complaints"
]

print(clean_data[converted_columns].dtypes)
display(clean_data[converted_columns].describe())

print("\nRemaining missing values:")
print(clean_data[["Gender", "Customer Lifetime Value"]].isna().sum())

Customer Lifetime Value      float64
Number of Open Complaints      Int64
dtype: object


,Customer Lifetime Value,Number of Open Complaints
count,9127.000000,9134.0
mean,7977.832132,0.383841
std,6810.190534,0.90911
min,1898.007675,0.0
25%,3980.630100,0.0
50%,5764.784200,0.0
75%,8931.228726,0.0
max,83325.381190,5.0



Remaining missing values:
Gender                     122
Customer Lifetime Value      7
dtype: int64


In [10]:
# Label missing gender values explicitly
clean_data["Gender"] = clean_data["Gender"].fillna("Unknown")

# Impute missing CLV values using the median
clv_median = clean_data["Customer Lifetime Value"].median()

clean_data["Customer Lifetime Value"] = (
    clean_data["Customer Lifetime Value"].fillna(clv_median)
)

print(f"CLV median used for imputation: {clv_median:,.2f}")

# Verify missing values and duplicates after standardization
print("\nRemaining missing values:")
print(clean_data.isna().sum())

print(f"\nFully duplicated rows: {clean_data.duplicated().sum()}")
print(f"Current dataset shape: {clean_data.shape}")

# Review all numerical columns for remaining anomalies
display(clean_data.describe().T)

CLV median used for imputation: 5,764.78

Remaining missing values:
Customer                     0
State                        0
Gender                       0
Education                    0
Customer Lifetime Value      0
Income                       0
Monthly Premium Auto         0
Number of Open Complaints    0
Policy Type                  0
Vehicle Class                0
Total Claim Amount           0
dtype: int64

Fully duplicated rows: 0
Current dataset shape: (9134, 11)


,count,mean,std,min,25%,50%,75%,max
Customer Lifetime Value,9134.0,7976.136125,6807.855681,1898.007675,3981.877092,5764.7842,8929.099793,83325.38119
Income,9134.0,37824.847055,30359.232933,0.0,0.0,34240.0,62446.5,99981.0
Monthly Premium Auto,9134.0,110.393146,581.471461,61.0,68.0,83.0,109.0,35354.0
Number of Open Complaints,9134.0,0.383841,0.90911,0.0,0.0,0.0,0.0,5.0
Total Claim Amount,9134.0,430.480412,289.617985,0.099007,266.961153,377.505619,546.050821,2893.239678


In [11]:
# Inspect the ten highest monthly premiums
columns_to_review = [
    "Customer",
    "Customer Lifetime Value",
    "Income",
    "Monthly Premium Auto"
]

display(
    clean_data.nlargest(10, "Monthly Premium Auto")[columns_to_review]
)

# Compare premium distributions across the original sources
premium_summary = pd.DataFrame({
    "Dataset 1": data_1["Monthly Premium Auto"].describe(),
    "Dataset 2": data_2["Monthly Premium Auto"].describe(),
    "Dataset 3": data_3["Monthly Premium Auto"].describe()
}).T

display(premium_summary[["count", "min", "50%", "75%", "max"]])

,Customer,Customer Lifetime Value,Income,Monthly Premium Auto
44,YD87931,4951.6561,46896.0,35354.0
856,QZ77637,11665.0978,84978.0,35353.0
747,JT11876,5435.7678,0.0,10202.0
1676,MP96373,5719.7857,54585.0,7575.0
114,SG20925,5185.7976,99428.0,6464.0
354,CI38330,2549.7861,72217.0,6464.0
741,AC40767,2497.8082,68041.0,6464.0
869,ER19995,17786.2778,99790.0,6464.0
1086,PE59368,5123.4850,72096.0,6464.0
1126,UB16654,4718.5585,15680.0,6464.0


,count,min,50%,75%,max
Dataset 1,1071.0,61.0,83.0,109.5,35354.0
Dataset 2,996.0,61.0,80.0,107.0,7575.0
Dataset 3,7070.0,61.0,83.0,109.0,298.0


### Data cleaning summary

-- Combined the three datasets after standardizing their column names.

-- Removed 2,937 completely empty rows and 3 fully duplicated rows.

-- Standardized inconsistent state, gender, and education labels.

-- Converted Customer Lifetime Value to a numeric type, dividing percentage-formatted values by 100 while preserving existing numeric values.

-- Converted Number of Open Complaints to an integer count by extracting the middle number from encoded values.

-- Replaced 122 missing gender values with "Unknown".

-- Imputed 7 missing Customer Lifetime Value values using the median, approximately 5,764.78.


The resulting dataset contains 9,134 rows and 11 columns, with no missing values or fully duplicated rows.

### Remaining data quality limitation

Monthly Premium Auto contains unusually high values in the first two source datasets, including 35,354 and 35,353. The maximum in the third source is 298. These differences suggest possible data errors, but the correct values cannot be established from the available information. The values were retained pending verification.

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [12]:
# Load the marketing customer analysis dataset
marketing_url = (
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/"
    "marketing_customer_analysis_clean.csv"
)

marketing_data = pd.read_csv(marketing_url)

# Inspect dimensions and column names
print(f"Dataset shape: {marketing_data.shape}")
print(f"Columns: {marketing_data.columns.tolist()}")

# Preview the dataset
display(marketing_data.head())

Dataset shape: (10910, 27)
Columns: ['unnamed:_0', 'customer', 'state', 'customer_lifetime_value', 'response', 'coverage', 'education', 'effective_to_date', 'employmentstatus', 'gender', 'income', 'location_code', 'marital_status', 'monthly_premium_auto', 'months_since_last_claim', 'months_since_policy_inception', 'number_of_open_complaints', 'number_of_policies', 'policy_type', 'policy', 'renew_offer_type', 'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size', 'vehicle_type', 'month']


,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2011-02-18,Employed,M,...,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,A,2
1,1,KX64629,California,2228.525238,No,Basic,College,2011-01-18,Unemployed,F,...,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,A,1
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2011-02-10,Employed,M,...,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A,2
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,1
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,2011-01-17,Medical Leave,F,...,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,A,1


In [13]:
# Inspect data types and non-null counts
marketing_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10910 entries, 0 to 10909
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   unnamed:_0                     10910 non-null  int64  
 1   customer                       10910 non-null  str    
 2   state                          10910 non-null  str    
 3   customer_lifetime_value        10910 non-null  float64
 4   response                       10910 non-null  str    
 5   coverage                       10910 non-null  str    
 6   education                      10910 non-null  str    
 7   effective_to_date              10910 non-null  str    
 8   employmentstatus               10910 non-null  str    
 9   gender                         10910 non-null  str    
 10  income                         10910 non-null  int64  
 11  location_code                  10910 non-null  str    
 12  marital_status                 10910 non-null  str    
 1

In [17]:
# Use monthly premiums as a proxy for monthly premium revenue
revenue_by_channel = marketing_data.pivot_table(
    index="sales_channel",
    values="monthly_premium_auto",
    aggfunc="sum"
)

# Rename the metric and sort from highest to lowest
revenue_by_channel = revenue_by_channel.rename(
    columns={"monthly_premium_auto": "total_monthly_premiums"}
).sort_values(
    by="total_monthly_premiums",
    ascending=False
)

# Display totals with two decimal places
with pd.option_context("display.float_format", "{:,.2f}".format):
    display(revenue_by_channel.astype(float).round(2))

,total_monthly_premiums
sales_channel,
Agent,"386,335.00"
Branch,"280,953.00"
Call Center,"197,970.00"
Web,"151,511.00"


### Sales channel analysis

-- Agent has the highest total monthly premiums at 386,335.00, followed by Branch at 280,953.00, Call Center at 197,970.00, and Web at 151,511.00.

-- Agent therefore leads under the selected revenue proxy, while Web has the lowest total. These totals reflect both the number of records and their premium amounts; they do not directly measure profitability.

-- Monthly premiums are used as a proxy because the dataset has no explicit revenue column. The observed channels include Agent, while Mail, mentioned in the instructions, does not appear in the resulting table.

In [18]:
# Calculate average customer lifetime value by gender and education
average_clv_by_gender_education = marketing_data.pivot_table(
    index="gender",
    columns="education",
    values="customer_lifetime_value",
    aggfunc="mean"
)

# Display averages with two decimal places
# Display average CLV with two decimal places
with pd.option_context("display.float_format", "{:,.2f}".format):
    display(average_clv_by_gender_education.round(2))

education,Bachelor,College,Doctor,High School or Below,Master
gender,,,,,
F,"7,874.27","7,748.82","7,328.51","8,675.22","8,157.05"
M,"7,703.60","8,052.46","7,415.33","8,149.69","8,168.83"


### Average customer lifetime value by gender and education

-- Female customers with High School or Below have the highest average CLV among the displayed groups, at 8,675.22. For male customers, the highest average is among those with a Master's degree, at 8,168.83.

-- Doctorate holders have the lowest average CLV within both genders: 7,328.51 for female customers and 7,415.33 for male customers.

-- Gender differences vary by educational level. Female customers have a higher average CLV in the Bachelor and High School or Below groups, while male customers have a higher average in the College, Doctor, and Master groups.

These descriptive results do not show a consistent increase in average CLV with higher educational attainment.

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [19]:
# Sum complaints by policy type and month in long format
complaints_by_policy_month = (
    marketing_data
    .groupby(["policy_type", "month"], as_index=False)[
        "number_of_open_complaints"
    ]
    .sum()
    .rename(columns={"number_of_open_complaints": "total_complaints"})
    .sort_values(["policy_type", "month"])
    .reset_index(drop=True)
)

display(complaints_by_policy_month)

# Identify the month or months with the most complaints for each policy type
maximum_by_policy = complaints_by_policy_month.groupby(
    "policy_type"
)["total_complaints"].transform("max")

highest_complaint_months = complaints_by_policy_month.loc[
    complaints_by_policy_month["total_complaints"] == maximum_by_policy
].reset_index(drop=True)

print("\nMonths with the highest complaint totals by policy type:")
display(highest_complaint_months)

,policy_type,month,total_complaints
0,Corporate Auto,1,443.434952
1,Corporate Auto,2,385.208135
2,Personal Auto,1,1727.605722
3,Personal Auto,2,1453.684441
4,Special Auto,1,87.074049
5,Special Auto,2,95.226817



Months with the highest complaint totals by policy type:


,policy_type,month,total_complaints
0,Corporate Auto,1,443.434952
1,Personal Auto,1,1727.605722
2,Special Auto,2,95.226817


### Complaints by policy type and month

-- Corporate Auto has its highest complaint total in January (month 1), at approximately 443.43, compared with 385.21 in February.

-- Personal Auto also peaks in January, at approximately 1,727.61, compared with 1,453.68 in February.

-- Special Auto peaks in February (month 2), at approximately 95.23, compared with 87.07 in January.


Personal Auto has the highest absolute complaint total in both months. However, comparing complaint rates would require accounting for the number of policies in each group.

### Data limitation

The supplied dataset produces fractional complaint totals. These results represent sums of the stored values rather than exact observed counts of complaint events. Fractional values may reflect prior preprocessing or imputation, but their origin has not been verified.